# Do Shooting Profiles Travel?
### Shot Selection, Shot Quality, and Performance Adaptation Between the NBA and EuroLeague

---

This notebook reproduces every statistic reported in the paper from the CSV
files in `../data/`. No database access is required.

| Section | Content |
|---|---|
| 1 | Setup and data loading |
| 2 | Spatial expected shot value (sxSV) model performance |
| 3 | Sample construction |
| 4 | Included vs excluded episodes |
| 5 | Within-player pre-post changes (primary shot-coordinate sample) |
| 6 | Field-goal-attempt threshold sensitivity |
| 7 | Matched-control comparison of changes |
| 8 | Dependence robustness |
| 9 | Direct vs one-season-gap transitions |
| 10 | Playing time, shot volume and usage |
| 11 | Player-season shooting profiles by league |
| 12 | Profile transferability |
| 13 | Repeated-measures models |
| 14 | Exploratory analyses |
| 15 | Paper statistics extraction |

**sxSV** is estimated with two specifications throughout: a **harmonised** one
using the same predictors in both leagues (shot distance, shot angle, shot
zone, shot value) and the **league-specific** one used in the submitted
manuscript, which adds the limited context each feed exposes. Both are
out-of-fold predictions from five-fold `GroupKFold` grouped by player, fitted
on the full shot population of each league.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, mannwhitneyu, chi2_contingency

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)

DATA = "../data"
RNG_SEED, N_BOOT = 42, 5000
SPECS = ["harmonised", "league_specific"]
THRESHOLDS = (50, 100, 150, 200)
PRIMARY = 100

players = pd.read_csv(f"{DATA}/players.csv")
episodes = pd.read_csv(f"{DATA}/migration_episodes.csv")
panel = pd.read_csv(f"{DATA}/player_season_boxscore.csv")
nba_ctrl = pd.read_csv(f"{DATA}/control_pool_nba.csv")
shots = pd.read_csv(f"{DATA}/shot_level_scored.csv")
model_metrics = pd.read_csv(f"{DATA}/sxsv_model_metrics.csv")
calib = pd.read_csv(f"{DATA}/sxsv_calibration_bins.csv")

panel["yr"] = panel["season"].str[:4].astype(int)
episodes["move_id"] = (episodes.player_id.astype(str) + "|" + episodes.direction
                       + "|" + episodes.destination_season)

print(f"players                 {len(players):>8,}")
print(f"migration episodes      {len(episodes):>8,}   ({episodes.player_id.nunique()} players)")
print(f"player-season box score {len(panel):>8,}")
print(f"NBA control pool        {len(nba_ctrl):>8,}")
print(f"shot-level records      {len(shots):>8,}")

players                    3,075
migration episodes           589   (283 players)
player-season box score   10,541
NBA control pool           8,180
shot-level records       130,434


In [2]:
def o_d_league(direction):
    """Origin and destination league for a migration direction."""
    return ("NBA", "EuroLeague") if direction == "NBA_to_EL" else ("EuroLeague", "NBA")


def boot_ci(d, clusters=None, n=N_BOOT, seed=RNG_SEED):
    """Percentile bootstrap of the mean.

    clusters=None -> ordinary resampling of episodes.
    clusters given -> resample PLAYERS with replacement, carrying all of a
    player's episodes together (player-clustered bootstrap).
    """
    d = np.asarray(d, float)
    if clusters is not None:
        clusters = np.asarray(clusters)
        keep = np.isfinite(d)
        d, clusters = d[keep], clusters[keep]
    else:
        d = d[np.isfinite(d)]
    if len(d) < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    if clusters is None:
        means = rng.choice(d, (n, len(d)), replace=True).mean(axis=1)
    else:
        groups = [d[clusters == c] for c in np.unique(clusters)]
        means = np.empty(n)
        for i in range(n):
            pick = rng.integers(0, len(groups), len(groups))
            means[i] = np.concatenate([groups[j] for j in pick]).mean()
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def rank_biserial(d):
    d = np.asarray(d, float); d = d[np.isfinite(d)]
    pos, neg = (d > 0).sum(), (d < 0).sum()
    return (pos - neg) / max(pos + neg, 1)


def smd(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    sp = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan


print("Helpers ready.")

Helpers ready.


## 2. Spatial expected shot value (sxSV) model performance

Out-of-fold performance on the full shot population of each league. The
population shot files (about 4.0 million NBA and 0.54 million EuroLeague
attempts) are too large to distribute here; the metrics and reliability bins
below are the model outputs those fits produced, and the shot-level file in
`../data/` already carries the resulting out-of-fold predictions for every
migrant attempt used downstream.

Expected calibration error uses **12 equal-width bins on [0, 1]**, weighted by
bin occupancy.

In [3]:
print("Out-of-fold sxSV model performance\n")
print(model_metrics.round(4).to_string(index=False))

print("\n\nReliability (12 quantile bins) - predicted vs observed make rate\n")
for (spec, lg), g in calib.groupby(["spec", "league"]):
    gap = (g.pred_mean - g.obs_rate).abs()
    print(f"  {spec:<16} {lg:<11} predicted {g.pred_mean.min():.3f}-{g.pred_mean.max():.3f}"
          f"   max |pred - obs| = {gap.max():.4f}")

Out-of-fold sxSV model performance

           spec     league                                               features       N  n_players    AUC  LogLoss  Brier    ECE
     harmonised        NBA                         dist_m,angle,zone_h,shot_value 4010977       2470 0.6292   0.6561 0.2321 0.0011
     harmonised EuroLeague                         dist_m,angle,zone_h,shot_value  540415       2057 0.6506   0.6361 0.2238 0.0010
league_specific        NBA                  dist_m,angle,zone_h,shot_value,period 4010977       2470 0.6299   0.6560 0.2320 0.0010
league_specific EuroLeague dist_m,angle,zone_h,shot_value,fastbreak,second_chance  540415       2057 0.6873   0.5863 0.2059 0.0010


Reliability (12 quantile bins) - predicted vs observed make rate

  harmonised       EuroLeague  predicted 0.294-0.849   max |pred - obs| = 0.0043
  harmonised       NBA         predicted 0.329-0.753   max |pred - obs| = 0.0034
  league_specific  EuroLeague  predicted 0.276-0.999   max |pred - obs| = 0.004

## 3. Sample construction

A **migration episode** is a transition from an eligible origin season in one
league to an eligible destination season in the other. Direct (gap 0) and
one-season-gap (gap 1) transitions are retained.

Because both gaps are retained, the same destination season can enter twice,
paired with two different origin seasons. A **distinct move** is therefore
defined as a unique player x direction x destination season.

In [4]:
psl_count = shots.groupby(["player_id", "league", "season"]).size().to_dict()


def fga_min_side(r):
    o_lg, d_lg = o_d_league(r.direction)
    return min(psl_count.get((r.player_id, o_lg, r.origin_season), 0),
               psl_count.get((r.player_id, d_lg, r.destination_season), 0))


episodes["fga_min_side"] = [fga_min_side(r) for r in episodes.itertuples()]

flow = [{"stage": "Players in cross-league database", "n": len(players),
         "NBA_to_EL": None, "EL_to_NBA": None},
        {"stage": "Cross-league migrants",
         "n": int(players.is_cross_league_migrant.sum()),
         "NBA_to_EL": None, "EL_to_NBA": None}]


def frow(stage, sub):
    return {"stage": stage, "n": len(sub),
            "NBA_to_EL": int((sub.direction == "NBA_to_EL").sum()),
            "EL_to_NBA": int((sub.direction == "EL_to_NBA").sum()),
            "players": sub.player_id.nunique(),
            "distinct_moves": sub.move_id.nunique()}


flow.append(frow("Valid migration episodes (gap 0 or 1)", episodes))
flow.append(frow("... with shot-coordinate data on both sides",
                 episodes[episodes.fga_min_side > 0]))
for thr in THRESHOLDS:
    flow.append(frow(f"... passing >={thr} FGA on both sides",
                     episodes[episodes.fga_min_side >= thr]))
flow_df = pd.DataFrame(flow)
print("Sample flow\n")
print(flow_df.to_string(index=False))

Sample flow

                                      stage    n  NBA_to_EL  EL_to_NBA  players  distinct_moves
           Players in cross-league database 3075        NaN        NaN      NaN             NaN
                      Cross-league migrants  402        NaN        NaN      NaN             NaN
      Valid migration episodes (gap 0 or 1)  589      346.0      243.0    283.0           418.0
... with shot-coordinate data on both sides   83       46.0       37.0     32.0            53.0
         ... passing >=50 FGA on both sides   46       24.0       22.0     16.0            26.0
        ... passing >=100 FGA on both sides   31       12.0       19.0     11.0            19.0
        ... passing >=150 FGA on both sides   19        4.0       15.0      9.0            12.0
        ... passing >=200 FGA on both sides   12        2.0       10.0      6.0             7.0


In [5]:
reasons = []
for r in episodes.itertuples():
    if r.fga_min_side >= PRIMARY:
        continue
    o_lg, d_lg = o_d_league(r.direction)
    pre = psl_count.get((r.player_id, o_lg, r.origin_season), 0)
    post = psl_count.get((r.player_id, d_lg, r.destination_season), 0)
    if pre == 0 and post == 0:
        reason = "no shot-coordinate data on either side"
    elif min(pre, post) == 0:
        reason = "no shot-coordinate data on one side"
    else:
        reason = f"shot data on both sides but <{PRIMARY} FGA"
    reasons.append({"direction": r.direction, "reason": reason})
er = pd.DataFrame(reasons).groupby(["reason", "direction"]).size().unstack(fill_value=0)
er["total"] = er.sum(axis=1)
print(f"Reasons for exclusion from the primary (>={PRIMARY} FGA) sample\n")
print(er.to_string())

Reasons for exclusion from the primary (>=100 FGA) sample

direction                               EL_to_NBA  NBA_to_EL  total
reason                                                             
no shot-coordinate data on either side         92        108    200
no shot-coordinate data on one side           114        192    306
shot data on both sides but <100 FGA           18         34     52


## 4. Included vs excluded episodes

Do the episodes that reach the primary threshold differ from those that do not?

In [6]:
pl_idx = players.set_index("player_id")
pan_idx = panel.set_index(["player_id", "league", "season"])

BOX_VARS = ["games_played", "minutes_per_game", "fga_per_game", "efg_pct",
            "three_pt_pct", "points_per_game"]


def season_row(pid, league, season):
    try:
        row = pan_idx.loc[(pid, league, season)]
    except KeyError:
        return None
    return row.iloc[0] if isinstance(row, pd.DataFrame) else row


recs = []
for r in episodes.itertuples():
    o_lg, _ = o_d_league(r.direction)
    o = season_row(r.player_id, o_lg, r.origin_season)
    y = int(r.origin_season[:4]) - 1
    prior = season_row(r.player_id, o_lg, f"{y}/{str(y + 1)[-2:]}")
    bio = pl_idx.loc[r.player_id]
    rec = {"player_id": r.player_id, "direction": r.direction, "gap": r.seasons_gap,
           "age": r.player_age_at_migration, "position": bio["position"],
           "height_cm": bio["height_cm"],
           "included": bool(r.fga_min_side >= PRIMARY)}
    for v in BOX_VARS:
        rec[v] = float(o[v]) if o is not None and pd.notna(o.get(v)) else np.nan
    for v in ["fga_per_game", "efg_pct", "three_pt_pct"]:
        rec[f"prior_{v}"] = (float(prior[v]) if prior is not None
                             and pd.notna(prior.get(v)) else np.nan)
    recs.append(rec)
epi_cov = pd.DataFrame(recs)

CONT = ["age", "height_cm"] + BOX_VARS + ["prior_fga_per_game", "prior_efg_pct",
                                          "prior_three_pt_pct"]
inc, exc = epi_cov[epi_cov.included], epi_cov[~epi_cov.included]
rows = []
for v in CONT:
    a, b = inc[v].dropna(), exc[v].dropna()
    p = mannwhitneyu(a, b).pvalue if len(a) >= 3 and len(b) >= 3 else np.nan
    rows.append({"variable": v, "n_incl": len(a), "incl_mean": a.mean(),
                 "incl_sd": a.std(ddof=1), "n_excl": len(b), "excl_mean": b.mean(),
                 "excl_sd": b.std(ddof=1), "SMD": smd(a, b), "MWU_p": p})
print(f"Included (n={len(inc)}) vs excluded (n={len(exc)}) episodes\n")
print(pd.DataFrame(rows).round(4).to_string(index=False))

ct = pd.crosstab(epi_cov["position"], epi_cov["included"])
print(f"\nPosition distribution (chi-square p = {chi2_contingency(ct.values)[1]:.4f})\n")
print(ct.rename(columns={False: "excluded", True: "included"}).to_string())

Included (n=31) vs excluded (n=558) episodes

          variable  n_incl  incl_mean  incl_sd  n_excl  excl_mean  excl_sd     SMD  MWU_p
               age      31    28.6452   3.5360     558    26.3889   3.8159  0.6133 0.0003
         height_cm      31   199.8387   9.1837     554   201.8520   9.3577 -0.2172 0.2456
      games_played      31    36.6129  14.5457     558    25.8297  20.9035  0.5988 0.0001
  minutes_per_game      31    20.8078   5.7819     469    15.3919   8.2719  0.7589 0.0001
      fga_per_game      31     6.8539   2.7027     469     4.8252   3.0870  0.6992 0.0001
           efg_pct      31     0.5108   0.0684     553     0.4685   0.1342  0.3974 0.0379
      three_pt_pct      31    34.9010   6.9815     500    25.8127  18.1262  0.6617 0.0035
   points_per_game      31     8.6226   3.8949     469     5.9062   4.4194  0.6521 0.0001
prior_fga_per_game      21     7.0681   2.2685     308     5.4159   2.9239  0.6314 0.0045
     prior_efg_pct      22     0.5196   0.0554     336

## 5. Within-player pre-post changes

Player-season shooting features are rebuilt from the shot-level file, then
paired across each migration episode.

In [7]:
PAIRED_METRICS = ["efg", "pps", "sxsv", "smoe", "three_rate", "mean_dist",
                  "pps_over_sxsv"]


def psl_features(min_fga, spec):
    """Player-season-league features at a given minimum-FGA threshold."""
    s = shots
    g = s.groupby(["player_id", "league", "season"])
    d = g.agg(fga=("made", "size"), fgm=("made", "sum"), pps=("pts", "mean"),
              sxsv=(f"sxsv_{spec}", "mean"), smoe=(f"smoe_{spec}", "mean"),
              mean_dist=("dist_m", "mean"),
              three_rate=("shot_value", lambda v: float((v == 3).mean()))).reset_index()
    m3 = (s.assign(m3=((s.shot_value == 3) & (s.made == 1)).astype(int))
          .groupby(["player_id", "league", "season"])["m3"].sum().rename("fg3m"))
    d = d.merge(m3, on=["player_id", "league", "season"], how="left")
    d["fg3m"] = d["fg3m"].fillna(0)
    d["efg"] = (d["fgm"] + 0.5 * d["fg3m"]) / d["fga"]
    d["pps_over_sxsv"] = d["pps"] - d["sxsv"]
    return d[d.fga >= min_fga].set_index(["player_id", "league", "season"])


def build_pairs(min_fga, spec):
    pk = psl_features(min_fga, spec)
    out = []
    for r in episodes.itertuples():
        o_lg, d_lg = o_d_league(r.direction)
        try:
            fo = pk.loc[(r.player_id, o_lg, r.origin_season)]
            fd = pk.loc[(r.player_id, d_lg, r.destination_season)]
        except KeyError:
            continue
        if isinstance(fo, pd.DataFrame) or isinstance(fd, pd.DataFrame):
            continue
        rec = {"player_id": r.player_id, "move_id": r.move_id,
               "direction": r.direction, "gap": r.seasons_gap,
               "fga_pre": fo["fga"], "fga_post": fd["fga"]}
        for m in PAIRED_METRICS:
            rec[f"{m}_pre"], rec[f"{m}_post"] = fo[m], fd[m]
            rec[f"d_{m}"] = fd[m] - fo[m]
        out.append(rec)
    return pd.DataFrame(out)


def summarise(df, label, spec, metrics=("efg", "pps", "sxsv", "smoe"), cluster=False):
    rows = []
    for m in metrics:
        d = df[f"d_{m}"].dropna()
        if len(d) < 3:
            continue
        cl = df.loc[d.index, "player_id"].values if cluster else None
        lo, hi = boot_ci(d.values, clusters=cl)
        try:
            p = wilcoxon(d.values).pvalue
        except ValueError:
            p = np.nan
        rows.append({"spec": spec, "group": label, "metric": m, "n": len(d),
                     "n_players": df.loc[d.index, "player_id"].nunique(),
                     "mean_pre": df[f"{m}_pre"].mean(),
                     "mean_post": df[f"{m}_post"].mean(),
                     "mean_delta": d.mean(), "ci_lo": lo, "ci_hi": hi,
                     "rank_biserial": rank_biserial(d.values), "wilcoxon_p": p})
    return rows


pairs = {(spec, thr): build_pairs(thr, spec) for spec in SPECS for thr in THRESHOLDS}
primary = pairs[("harmonised", PRIMARY)]
print(f"Primary sample: {len(primary)} episodes, {primary.player_id.nunique()} players, "
      f"{primary.move_id.nunique()} distinct moves")
print(primary.direction.value_counts().to_string())

Primary sample: 31 episodes, 11 players, 19 distinct moves
direction
EL_to_NBA    19
NBA_to_EL    12


In [8]:
rows = []
for spec in SPECS:
    pp = pairs[(spec, PRIMARY)]
    rows += summarise(pp, "ALL", spec, metrics=PAIRED_METRICS)
    for dirn in ["EL_to_NBA", "NBA_to_EL"]:
        rows += summarise(pp[pp.direction == dirn], dirn, spec, metrics=PAIRED_METRICS)
prepost = pd.DataFrame(rows)

# Benjamini-Hochberg FDR across features, within each specification and group.
from statsmodels.stats.multitest import multipletests
prepost["p_fdr"] = np.nan
for (spec, grp), idx in prepost.groupby(["spec", "group"]).groups.items():
    p = prepost.loc[idx, "wilcoxon_p"]
    ok = p.notna()
    if ok.sum():
        prepost.loc[p[ok].index, "p_fdr"] = multipletests(p[ok], method="fdr_bh")[1]
prepost["sig_fdr"] = np.where(prepost["p_fdr"] < 0.05, "*", "")

print(f"Pre-post change, primary (>={PRIMARY} FGA) sample - both sxSV specifications")
print("p_fdr is Benjamini-Hochberg corrected across features within each group\n")
print(prepost.round(4).to_string(index=False))

Pre-post change, primary (>=100 FGA) sample - both sxSV specifications
p_fdr is Benjamini-Hochberg corrected across features within each group

           spec     group        metric  n  n_players  mean_pre  mean_post  mean_delta   ci_lo   ci_hi  rank_biserial  wilcoxon_p  p_fdr sig_fdr
     harmonised       ALL           efg 31         11    0.5157     0.4930     -0.0226 -0.0489  0.0038        -0.0968      0.1757 0.6070        
     harmonised       ALL           pps 31         11    1.0313     0.9860     -0.0453 -0.0978  0.0076        -0.0968      0.1757 0.6070        
     harmonised       ALL          sxsv 31         11    1.0345     1.0253     -0.0092 -0.0312  0.0111        -0.0968      0.5294 0.7412        
     harmonised       ALL          smoe 31         11    0.0008    -0.0143     -0.0151 -0.0354  0.0045        -0.0968      0.3271 0.6070        
     harmonised       ALL    three_rate 31         11    0.4653     0.4415     -0.0238 -0.0899  0.0392         0.0968      0.8092 0

In [9]:
key = ["player_id", "direction", "gap"]
a = pairs[("harmonised", PRIMARY)][key + ["sxsv_pre", "sxsv_post", "d_sxsv"]]
b = pairs[("league_specific", PRIMARY)][key + ["sxsv_pre", "sxsv_post", "d_sxsv"]]
m = a.merge(b, on=key, suffixes=("_harm", "_lsp"), validate="one_to_one")
agree = pd.DataFrame([{
    "n_episodes": len(m),
    "r_sxSV_pre": m.sxsv_pre_harm.corr(m.sxsv_pre_lsp),
    "r_sxSV_post": m.sxsv_post_harm.corr(m.sxsv_post_lsp),
    "r_delta_sxSV": m.d_sxsv_harm.corr(m.d_sxsv_lsp),
    "spearman_delta": m.d_sxsv_harm.corr(m.d_sxsv_lsp, method="spearman"),
    "mean_delta_harmonised": m.d_sxsv_harm.mean(),
    "mean_delta_league_specific": m.d_sxsv_lsp.mean(),
    "sign_agreement_pct": 100 * (np.sign(m.d_sxsv_harm) == np.sign(m.d_sxsv_lsp)).mean()}])
print("Comparability of the two sxSV specifications\n")
print(agree.round(4).to_string(index=False))

Comparability of the two sxSV specifications

 n_episodes  r_sxSV_pre  r_sxSV_post  r_delta_sxSV  spearman_delta  mean_delta_harmonised  mean_delta_league_specific  sign_agreement_pct
         31      0.9361        0.978        0.8157          0.8395                -0.0092                     -0.0102              83.871


## 6. Field-goal-attempt threshold sensitivity

In [10]:
rows = []
for spec in SPECS:
    for thr in THRESHOLDS:
        pp = pairs[(spec, thr)]
        if not len(pp):
            continue
        for r in summarise(pp, "ALL", spec):
            r["threshold"] = thr; rows.append(r)
        for dirn in ["EL_to_NBA", "NBA_to_EL"]:
            for r in summarise(pp[pp.direction == dirn], dirn, spec):
                r["threshold"] = thr; rows.append(r)
sens = pd.DataFrame(rows)
cols = ["spec", "threshold", "group", "metric", "n", "n_players", "mean_delta",
        "ci_lo", "ci_hi", "rank_biserial", "wilcoxon_p"]
print("Pre-post change by FGA threshold (mean change, bootstrap 95% CI)\n")
print(sens[cols].round(4).to_string(index=False))

Pre-post change by FGA threshold (mean change, bootstrap 95% CI)

           spec  threshold     group metric  n  n_players  mean_delta   ci_lo   ci_hi  rank_biserial  wilcoxon_p
     harmonised         50       ALL    efg 46         16     -0.0067 -0.0313  0.0178        -0.0870      0.5807
     harmonised         50       ALL    pps 46         16     -0.0133 -0.0625  0.0356        -0.0870      0.5807
     harmonised         50       ALL   sxsv 46         16     -0.0260 -0.0565 -0.0000        -0.2174      0.1634
     harmonised         50       ALL   smoe 46         16      0.0037 -0.0138  0.0201         0.0870      0.4351
     harmonised         50 EL_to_NBA    efg 22         11     -0.0309 -0.0690  0.0082        -0.3636      0.1129
     harmonised         50 EL_to_NBA    pps 22         11     -0.0617 -0.1379  0.0164        -0.3636      0.1129
     harmonised         50 EL_to_NBA   sxsv 22         11      0.0012 -0.0301  0.0308         0.0000      0.9493
     harmonised         50 EL_

## 7. Matched-control comparison of changes

Each migrant is matched to the five nearest non-migrant stayers in the origin
league (standardised distance on the matching covariates, same origin season
+/- 1). The estimate is the migrant's cross-league change minus the matched
stayers' within-league change.

The EuroLeague control pool carries minutes, age, position and experience, so
the matching can be extended there. The NBA control pool carries only shooting
volume and efficiency, so those covariates are unavailable in that direction.

In [11]:
OUTCOMES = ["efg_pct", "three_pt_pct", "fga_per_game"]
K_MATCH = 5

mig_ids = set(players.loc[players.is_cross_league_migrant == 1, "player_id"])
byear = players.set_index("player_id")["birth_year"].to_dict()
pos_map = players.set_index("player_id")["position"].to_dict()

el_p = panel[panel.league == "EuroLeague"].drop_duplicates(["player_id", "season"]).copy()
nba_p = panel[panel.league == "NBA"].drop_duplicates(["player_id", "season"]).copy()
el_box = el_p.set_index(["player_id", "season"])
nba_box = nba_p.set_index(["player_id", "season"])

nbac = nba_ctrl.rename(columns={"nba_player_id": "player_id"}).copy()
nbac["yr"] = nbac["season"].str[:4].astype(int)
nbac = nbac.drop_duplicates(["player_id", "season"])


def experience_map(p):
    p = p.sort_values(["player_id", "yr"])
    return {(pid, yr): e for (pid, yr), e in
            zip(zip(p.player_id, p.yr), p.groupby("player_id").cumcount())}


el_exp = experience_map(el_p)
nba_exp = experience_map(nba_p[nba_p.season >= "2007/08"])
nbac["experience"] = nbac.sort_values("yr").groupby("player_id").cumcount()


def build_pool(p, id_col="player_id"):
    nd = {}
    for pid, g in p.sort_values([id_col, "yr"]).groupby(id_col):
        g = g.sort_values("yr")
        for i in range(len(g) - 1):
            if g.iloc[i + 1]["yr"] - g.iloc[i]["yr"] == 1:
                nd[(pid, g.iloc[i]["yr"])] = {f: g.iloc[i + 1][f] - g.iloc[i][f]
                                              for f in OUTCOMES}
    keep = p.apply(lambda r: (r[id_col], r["yr"]) in nd, axis=1)
    return p[keep].dropna(subset=OUTCOMES).copy(), nd


el_pool, el_nd = build_pool(el_p[~el_p.player_id.isin(mig_ids)])
el_pool["age"] = el_pool["yr"] - el_pool["player_id"].map(byear)
el_pool["position"] = el_pool["player_id"].map(pos_map)
el_pool["experience"] = [el_exp.get((p_, y), np.nan)
                         for p_, y in zip(el_pool.player_id, el_pool.yr)]
nba_pool, nba_nd = build_pool(nbac)

COV_SETS = {
    "base (paper)": ["fga_per_game", "efg_pct", "three_pt_pct"],
    "+ minutes": ["fga_per_game", "efg_pct", "three_pt_pct", "minutes_per_game"],
    "+ minutes + age + experience": ["fga_per_game", "efg_pct", "three_pt_pct",
                                     "minutes_per_game", "age", "experience"],
}


def matched_did(direction, covs, exact_position=False, eps=None):
    if direction == "EL_to_NBA":
        o_box, d_box, pool, nd, o_lg = el_box, nba_box, el_pool, el_nd, "EuroLeague"
    else:
        o_box, d_box, pool, nd, o_lg = nba_box, el_box, nba_pool, nba_nd, "NBA"
    requested = list(covs)
    covs = [c for c in covs if c in pool.columns and pool[c].notna().mean() > 0.2]
    dropped = [c for c in requested if c not in covs]
    pool_u = pool.dropna(subset=covs)
    if len(pool_u) < K_MATCH:
        return None
    scales = np.array([pool_u[c].std(ddof=0) + 1e-9 for c in covs])
    pool_means = pool_u[covs].mean().values

    eps = episodes if eps is None else eps
    eps = eps[eps.direction == direction]
    md_, cd_, bm, bc, pids = [], [], [], [], []
    for r in eps.itertuples():
        try:
            o, dst = o_box.loc[(r.player_id, r.origin_season)], d_box.loc[
                (r.player_id, r.destination_season)]
        except KeyError:
            continue
        if isinstance(o, pd.DataFrame) or isinstance(dst, pd.DataFrame):
            continue
        Y = int(r.origin_season[:4])
        by = byear.get(r.player_id, np.nan)
        exp_src = el_exp if o_lg == "EuroLeague" else nba_exp
        mv = {"fga_per_game": o.get("fga_per_game"), "efg_pct": o.get("efg_pct"),
              "three_pt_pct": o.get("three_pt_pct"),
              "minutes_per_game": o.get("minutes_per_game"),
              "age": Y - by if pd.notna(by) else np.nan,
              "experience": exp_src.get((r.player_id, Y), np.nan)}
        vm = np.array([mv.get(c, np.nan) for c in covs], float)
        if not np.isfinite(vm).all():
            continue
        cp = pool_u[(pool_u.yr.between(Y - 1, Y + 1)) & (pool_u.player_id != r.player_id)]
        if exact_position and "position" in cp.columns:
            pp = pos_map.get(r.player_id)
            if pp and cp["position"].notna().any():
                cp2 = cp[cp["position"] == pp]
                if len(cp2) >= K_MATCH:
                    cp = cp2
        if len(cp) < K_MATCH:
            continue
        diff = (cp[covs].values - vm) / scales
        nn = cp.iloc[np.argsort(np.sqrt((diff ** 2).sum(axis=1)))[:K_MATCH]]
        md_.append({f: dst[f] - o[f] for f in OUTCOMES})
        cd_.append({f: np.mean([nd[(c.player_id, c.yr)][f] for c in nn.itertuples()])
                    for f in OUTCOMES})
        bm.append(vm); bc.append(nn[covs].mean().values); pids.append(r.player_id)
    if not md_:
        return None
    md_, cd_, pids = pd.DataFrame(md_), pd.DataFrame(cd_), np.array(pids)
    bm_a, bc_a = np.array(bm), np.array(bc)
    bal = pd.DataFrame({"covariate": covs, "migrant_mean": bm_a.mean(0),
                        "matched_control_mean": bc_a.mean(0),
                        "SMD_before": (bm_a.mean(0) - pool_means) / scales,
                        "SMD_after": (bm_a.mean(0) - bc_a.mean(0)) / scales})
    rows = []
    for f in OUTCOMES:
        v = (md_[f] - cd_[f]).values
        ok = np.isfinite(v)
        v, cl = v[ok], pids[ok]
        try:
            p = wilcoxon(v).pvalue
        except ValueError:
            p = np.nan
        lo, hi = boot_ci(v)
        lo_c, hi_c = boot_ci(v, clusters=cl)
        rows.append({"outcome": f, "n": len(v), "n_players": len(np.unique(cl)),
                     "migrant_delta": md_[f].mean(), "control_delta": cd_[f].mean(),
                     "estimate": v.mean(), "ci_lo": lo, "ci_hi": hi,
                     "ci_lo_clustered": lo_c, "ci_hi_clustered": hi_c,
                     "wilcoxon_p": p,
                     "covariates_used": "+".join(covs),
                     "covariates_dropped": "+".join(dropped) if dropped else "-"})
    return pd.DataFrame(rows), bal


did_rows, bal_rows = [], []
for direction in ["EL_to_NBA", "NBA_to_EL"]:
    for name, covs in COV_SETS.items():
        res = matched_did(direction, covs)
        if res is None:
            continue
        tab, bal = res
        tab.insert(0, "matching", name); tab.insert(0, "direction", direction)
        did_rows.append(tab)
        bal.insert(0, "matching", name); bal.insert(0, "direction", direction)
        bal_rows.append(bal)
did_tab = pd.concat(did_rows, ignore_index=True)
print("Matched-control comparison of changes under alternative matching covariates\n")
print(did_tab.round(4).to_string(index=False))

Matched-control comparison of changes under alternative matching covariates

direction                     matching      outcome   n  n_players  migrant_delta  control_delta  estimate    ci_lo   ci_hi  ci_lo_clustered  ci_hi_clustered  wilcoxon_p                                                   covariates_used   covariates_dropped
EL_to_NBA                 base (paper)      efg_pct 128         87        -0.0675        -0.0052   -0.0635  -0.0832 -0.0455          -0.0863          -0.0430      0.0000                                 fga_per_game+efg_pct+three_pt_pct                    -
EL_to_NBA                 base (paper) three_pt_pct 110         79        -8.4862         0.4269   -7.8111 -10.3765 -5.3287         -10.9086          -4.8067      0.0000                                 fga_per_game+efg_pct+three_pt_pct                    -
EL_to_NBA                 base (paper) fga_per_game  47         39        -2.9895        -0.8119   -2.8749  -3.7711 -2.0300          -3.9382          -1

In [12]:
print("Covariate balance (standardised mean differences)\n")
print(pd.concat(bal_rows, ignore_index=True).round(4).to_string(index=False))

Covariate balance (standardised mean differences)

direction                     matching        covariate  migrant_mean  matched_control_mean  SMD_before  SMD_after
EL_to_NBA                 base (paper)     fga_per_game        7.9526                7.7744      0.5878     0.0641
EL_to_NBA                 base (paper)          efg_pct        0.5342                0.5328      0.0442     0.0202
EL_to_NBA                 base (paper)     three_pt_pct       35.6742               35.8000     -0.0664    -0.0130
EL_to_NBA                    + minutes     fga_per_game        7.9526                7.7208      0.5878     0.0834
EL_to_NBA                    + minutes          efg_pct        0.5342                0.5321      0.0442     0.0300
EL_to_NBA                    + minutes     three_pt_pct       35.6742               35.7844     -0.0664    -0.0114
EL_to_NBA                    + minutes minutes_per_game       23.9912               24.0327      0.4771    -0.0067
EL_to_NBA + minutes + age + e

## 8. Dependence robustness

Some players contribute more than one episode, and gap-0 / gap-1 versions of
the same move share a destination season. Three restrictions are compared with
the full sample.

In [13]:
rep = (episodes.groupby("player_id").size().value_counts().sort_index()
       .rename_axis("episodes_per_player").reset_index(name="n_players"))
print(f"Box-score sample: {len(episodes)} episodes, {episodes.player_id.nunique()} "
      f"players, {episodes.move_id.nunique()} distinct moves\n")
print(rep.to_string(index=False))

rep2 = (primary.groupby("player_id").size().value_counts().sort_index()
        .rename_axis("episodes_per_player").reset_index(name="n_players"))
print(f"\nPrimary sample: {len(primary)} episodes, {primary.player_id.nunique()} "
      f"players, {primary.move_id.nunique()} distinct moves\n")
print(rep2.to_string(index=False))

Box-score sample: 589 episodes, 283 players, 418 distinct moves

 episodes_per_player  n_players
                   1        108
                   2         96
                   3         34
                   4         40
                   5          3
                   6          2

Primary sample: 31 episodes, 11 players, 19 distinct moves

 episodes_per_player  n_players
                   2          5
                   3          3
                   4          3


In [14]:
rows = []
for spec in SPECS:
    pp = pairs[(spec, PRIMARY)]
    one_move = pp.sort_values(["move_id", "gap"]).drop_duplicates("move_id", keep="first")
    one_player = pp.sort_values(["player_id", "gap"]).drop_duplicates("player_id",
                                                                     keep="first")
    rows += summarise(pp, "all episodes (iid bootstrap)", spec)
    rows += summarise(pp, "all episodes (player-clustered bootstrap)", spec, cluster=True)
    rows += summarise(one_move, "one episode per distinct move", spec)
    rows += summarise(one_player, "one episode per player", spec)
    for dirn in ["EL_to_NBA", "NBA_to_EL"]:
        rows += summarise(one_move[one_move.direction == dirn],
                          f"one per distinct move, {dirn}", spec)
dep = pd.DataFrame(rows)
print("Dependence robustness, primary sample\n")
print(dep[["spec", "group", "metric", "n", "n_players", "mean_delta", "ci_lo",
           "ci_hi", "wilcoxon_p"]].round(4).to_string(index=False))

Dependence robustness, primary sample

           spec                                     group metric  n  n_players  mean_delta   ci_lo   ci_hi  wilcoxon_p
     harmonised              all episodes (iid bootstrap)    efg 31         11     -0.0226 -0.0489  0.0038      0.1757
     harmonised              all episodes (iid bootstrap)    pps 31         11     -0.0453 -0.0978  0.0076      0.1757
     harmonised              all episodes (iid bootstrap)   sxsv 31         11     -0.0092 -0.0312  0.0111      0.5294
     harmonised              all episodes (iid bootstrap)   smoe 31         11     -0.0151 -0.0354  0.0045      0.3271
     harmonised all episodes (player-clustered bootstrap)    efg 31         11     -0.0226 -0.0507  0.0058      0.1757
     harmonised all episodes (player-clustered bootstrap)    pps 31         11     -0.0453 -0.1014  0.0116      0.1757
     harmonised all episodes (player-clustered bootstrap)   sxsv 31         11     -0.0092 -0.0362  0.0171      0.5294
     harm

In [15]:
rows = []
for direction in ["EL_to_NBA", "NBA_to_EL"]:
    one_move = episodes.sort_values(["move_id", "seasons_gap"]).drop_duplicates("move_id")
    one_player = episodes.sort_values(["player_id", "origin_season"]).drop_duplicates(
        "player_id")
    for lbl, e in [("all episodes", None), ("one episode per distinct move", one_move),
                   ("one episode per player", one_player)]:
        res = matched_did(direction, COV_SETS["base (paper)"], eps=e)
        if res is None:
            continue
        t = res[0].copy()
        t.insert(0, "sample", lbl); t.insert(0, "direction", direction)
        rows.append(t)
print("Matched-control estimates under the same restrictions\n")
print(pd.concat(rows, ignore_index=True)[
    ["direction", "sample", "outcome", "n", "n_players", "estimate", "ci_lo", "ci_hi",
     "ci_lo_clustered", "ci_hi_clustered", "wilcoxon_p"]].round(4).to_string(index=False))

Matched-control estimates under the same restrictions

direction                        sample      outcome   n  n_players  estimate    ci_lo   ci_hi  ci_lo_clustered  ci_hi_clustered  wilcoxon_p
EL_to_NBA                  all episodes      efg_pct 128         87   -0.0635  -0.0832 -0.0455          -0.0863          -0.0430      0.0000
EL_to_NBA                  all episodes three_pt_pct 110         79   -7.8111 -10.3765 -5.3287         -10.9086          -4.8067      0.0000
EL_to_NBA                  all episodes fga_per_game  47         39   -2.8749  -3.7711 -2.0300          -3.9382          -1.9004      0.0000
EL_to_NBA one episode per distinct move      efg_pct  82         82   -0.0755  -0.1022 -0.0508          -0.1021          -0.0509      0.0000
EL_to_NBA one episode per distinct move three_pt_pct  73         73   -8.4840 -11.9117 -5.2062         -11.9902          -5.0453      0.0000
EL_to_NBA one episode per distinct move fga_per_game  31         31   -2.4790  -3.5607 -1.4586     

## 9. Direct vs one-season-gap transitions

In [16]:
gap_counts = episodes.groupby(["seasons_gap", "direction"]).size().unstack(fill_value=0)
gap_counts["total"] = gap_counts.sum(axis=1)
print("Episode counts by season gap (box-score sample)\n")
print(gap_counts.to_string())

gp = primary.groupby(["gap", "direction"]).size().unstack(fill_value=0)
gp["total"] = gp.sum(axis=1)
print(f"\nEpisode counts by season gap (primary sample)\n")
print(gp.to_string())

rows = []
for spec in SPECS:
    pp = pairs[(spec, PRIMARY)]
    for g, lbl in [(0, "direct (gap 0)"), (1, "one-season gap (gap 1)")]:
        rows += summarise(pp[pp.gap == g], lbl, spec)
print("\nPre-post change by transition type, primary sample\n")
print(pd.DataFrame(rows)[["spec", "group", "metric", "n", "n_players", "mean_delta",
                          "ci_lo", "ci_hi", "wilcoxon_p"]].round(4).to_string(index=False))

Episode counts by season gap (box-score sample)

direction    EL_to_NBA  NBA_to_EL  total
seasons_gap                             
0                  149        182    331
1                   94        164    258

Episode counts by season gap (primary sample)

direction  EL_to_NBA  NBA_to_EL  total
gap                                   
0                 11          7     18
1                  8          5     13

Pre-post change by transition type, primary sample

           spec                  group metric  n  n_players  mean_delta   ci_lo   ci_hi  wilcoxon_p
     harmonised         direct (gap 0)    efg 18         11     -0.0081 -0.0475  0.0293      0.8986
     harmonised         direct (gap 0)    pps 18         11     -0.0161 -0.0951  0.0585      0.8986
     harmonised         direct (gap 0)   sxsv 18         11      0.0019 -0.0215  0.0249      1.0000
     harmonised         direct (gap 0)   smoe 18         11     -0.0081 -0.0384  0.0203      0.9323
     harmonised one-season gap

## 10. Playing time, shot volume and usage

Field-goal attempts per game can fall simply because playing time falls. Per
minute measures separate the two. The usage proxy is
`(FGA + 0.44 x FTA + turnovers) / minutes`; team possessions are not available.

In [17]:
panel["fta_per_game"] = panel["ft_attempts_total"] / panel["games_played"].replace(0, np.nan)
panel["fga_per_min"] = panel["fga_per_game"] / panel["minutes_per_game"].replace(0, np.nan)
panel["usage_proxy_per_min"] = (
    (panel["fga_per_game"] + 0.44 * panel["fta_per_game"] + panel["turnovers_per_game"])
    / panel["minutes_per_game"].replace(0, np.nan))
pan_idx = panel.set_index(["player_id", "league", "season"])

USAGE = ["minutes_per_game", "fga_per_game", "fga_per_min", "usage_proxy_per_min",
         "points_per_game", "efg_pct"]
rows = []
for r in episodes.itertuples():
    o_lg, d_lg = o_d_league(r.direction)
    fo, fd = season_row(r.player_id, o_lg, r.origin_season), season_row(
        r.player_id, d_lg, r.destination_season)
    if fo is None or fd is None:
        continue
    rec = {"player_id": r.player_id, "direction": r.direction, "gap": r.seasons_gap,
           "in_primary": bool(r.fga_min_side >= PRIMARY)}
    for v in USAGE:
        rec[f"{v}_pre"], rec[f"{v}_post"] = fo.get(v), fd.get(v)
        rec[f"d_{v}"] = (fd.get(v) - fo.get(v)) if pd.notna(fo.get(v)) and pd.notna(
            fd.get(v)) else np.nan
    rows.append(rec)
usage = pd.DataFrame(rows)


def usage_summary(df, label):
    out = []
    for v in USAGE:
        d = df[f"d_{v}"].dropna()
        if len(d) < 3:
            continue
        lo, hi = boot_ci(d.values)
        lo_c, hi_c = boot_ci(d.values, clusters=df.loc[d.index, "player_id"].values)
        out.append({"sample": label, "variable": v, "n": len(d),
                    "n_players": df.loc[d.index, "player_id"].nunique(),
                    "mean_pre": df[f"{v}_pre"].mean(), "mean_post": df[f"{v}_post"].mean(),
                    "mean_delta": d.mean(), "ci_lo": lo, "ci_hi": hi,
                    "ci_lo_clustered": lo_c, "ci_hi_clustered": hi_c,
                    "wilcoxon_p": wilcoxon(d.values).pvalue})
    return out


u = usage_summary(usage, "all episodes")
for dirn in ["EL_to_NBA", "NBA_to_EL"]:
    u += usage_summary(usage[usage.direction == dirn], dirn)
u += usage_summary(usage[usage.in_primary], "primary sample")
print("Playing time, shot volume and usage before and after migration\n")
print(pd.DataFrame(u).round(4).to_string(index=False))

Playing time, shot volume and usage before and after migration

        sample            variable   n  n_players  mean_pre  mean_post  mean_delta   ci_lo   ci_hi  ci_lo_clustered  ci_hi_clustered  wilcoxon_p
  all episodes    minutes_per_game 327        162   15.7277    18.3960      1.4047  0.0487  2.7153           0.1025           2.7175      0.0388
  all episodes        fga_per_game 327        162    4.9510     5.8863      0.5093 -0.0130  1.0081          -0.0035           1.0205      0.0531
  all episodes         fga_per_min 327        162    0.3111     0.3159      0.0061 -0.0045  0.0163          -0.0051           0.0169      0.6799
  all episodes usage_proxy_per_min 327        162    0.4152     0.4223      0.0093 -0.0032  0.0218          -0.0041           0.0229      0.2920
  all episodes     points_per_game 327        162    6.0746     7.3976      0.6866 -0.0964  1.4288          -0.0850           1.4675      0.0669
  all episodes             efg_pct 575        276    0.4707     0.

## 11. Player-season shooting profiles by league

Descriptive profile of every eligible player-season-league observation at the
primary threshold (manuscript Table 3).

In [18]:
def shot_entropy(zone_series):
    p = zone_series.value_counts(normalize=True)
    return float(-(p * np.log(p)).sum())


def build_profiles(min_fga=PRIMARY, spec="league_specific"):
    """Full shot-selection + value feature set per player-season-league."""
    recs = []
    for (pid, lg, se), g in shots.groupby(["player_id", "league", "season"]):
        if len(g) < min_fga:
            continue
        zc = g["zone_h"].value_counts(normalize=True)
        fga = len(g)
        recs.append({
            "player_id": pid, "league": lg, "season": se, "fga": fga,
            "rim_rate": zc.get("rim", 0.0),
            "paint_rate": zc.get("paint_non_rim", 0.0),
            "mid_rate": zc.get("mid_range", 0.0),
            "corner3_rate": zc.get("corner_3", 0.0),
            "abovebreak3_rate": zc.get("above_break_3", 0.0),
            "three_rate": float((g["shot_value"] == 3).mean()),
            "mean_dist": float(g["dist_m"].mean()),
            "sd_dist": float(g["dist_m"].std(ddof=0)),
            "zone_entropy": shot_entropy(g["zone_h"]),
            "laterality": float(g["x_m"].mean()),
            "efg": float((g["made"].sum()
                          + 0.5 * ((g["shot_value"] == 3) & (g["made"] == 1)).sum()) / fga),
            "pps": float(g["pts"].mean()),
            "xsv": float(g[f"sxsv_{spec}"].mean()),
            "make_over_exp": float(g[f"smoe_{spec}"].mean()),
        })
    d = pd.DataFrame(recs)
    d["xefg"] = d["xsv"] / 2
    d["pps_over_xsv"] = d["pps"] - d["xsv"]
    return d


prof = build_profiles()
PROFILE_COLS = ["fga", "rim_rate", "paint_rate", "mid_rate", "corner3_rate",
                "abovebreak3_rate", "three_rate", "mean_dist", "sd_dist",
                "zone_entropy", "efg", "pps", "xsv", "xefg", "make_over_exp",
                "pps_over_xsv"]
print(f"Eligible player-season-league observations (>={PRIMARY} FGA): {len(prof)}")
print(prof.league.value_counts().to_string())
print("\nMeans by league\n")
print(prof.groupby("league")[PROFILE_COLS].mean().round(4).T.to_string())

Eligible player-season-league observations (>=100 FGA): 428
league
EuroLeague    308
NBA           120

Means by league

league            EuroLeague       NBA
fga                 189.4188  441.3083
rim_rate              0.2080    0.3411
paint_rate            0.2433    0.1299
mid_rate              0.1192    0.2549
corner3_rate          0.0581    0.0790
abovebreak3_rate      0.3714    0.1951
three_rate            0.4294    0.2741
mean_dist             4.5622    3.9919
sd_dist               2.3555    2.6987
zone_entropy          1.2338    1.2594
efg                   0.5432    0.4987
pps                   1.0864    0.9975
xsv                   1.0614    1.0269
xefg                  0.5307    0.5134
make_over_exp         0.0103   -0.0130
pps_over_xsv          0.0250   -0.0294


## 12. Profile transferability

Pre-post correlation for each feature across the paired episodes, split into
spatial shot-selection features and efficiency/value features. This is the
direct test of the expectation that spatial tendencies travel better than
realised efficiency.

In [19]:
from scipy.stats import pearsonr, spearmanr

CORR_FEATURES = ["three_rate", "rim_rate", "mid_rate", "mean_dist", "zone_entropy",
                 "efg", "pps", "xsv", "make_over_exp"]
SPATIAL = {"three_rate", "rim_rate", "mid_rate", "mean_dist", "zone_entropy"}

prof_key = prof.set_index(["player_id", "league", "season"])
rows = []
for r in episodes.itertuples():
    o_lg, d_lg = o_d_league(r.direction)
    try:
        fo = prof_key.loc[(r.player_id, o_lg, r.origin_season)]
        fd = prof_key.loc[(r.player_id, d_lg, r.destination_season)]
    except KeyError:
        continue
    rec = {"player_id": r.player_id, "direction": r.direction}
    for f in CORR_FEATURES:
        rec[f"{f}__pre"], rec[f"{f}__post"] = fo[f], fd[f]
    rows.append(rec)
prof_pairs = pd.DataFrame(rows)
print(f"Paired episodes with complete profiles: {len(prof_pairs)}\n")

rows = []
for f in CORR_FEATURES:
    d = prof_pairs.dropna(subset=[f"{f}__pre", f"{f}__post"])
    r_p, p_p = pearsonr(d[f"{f}__pre"], d[f"{f}__post"])
    r_s, p_s = spearmanr(d[f"{f}__pre"], d[f"{f}__post"])
    rows.append({"feature": f,
                 "group": "spatial" if f in SPATIAL else "efficiency/value",
                 "n": len(d), "pearson_r": r_p, "pearson_p": p_p,
                 "spearman_r": r_s, "spearman_p": p_s})
corr = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
print(corr.round(4).to_string(index=False))
print("\nMean pre-post correlation by feature group")
print(corr.groupby("group")["pearson_r"].mean().round(4).to_string())

Paired episodes with complete profiles: 31

      feature            group  n  pearson_r  pearson_p  spearman_r  spearman_p
     mid_rate          spatial 31     0.7286     0.0000      0.4647      0.0084
          xsv efficiency/value 31     0.7089     0.0000      0.7457      0.0000
   three_rate          spatial 31     0.2661     0.1479      0.1843      0.3210
 zone_entropy          spatial 31     0.2164     0.2423      0.3008      0.1002
make_over_exp efficiency/value 31     0.1427     0.4438      0.1427      0.4438
     rim_rate          spatial 31     0.0535     0.7748      0.0445      0.8122
    mean_dist          spatial 31     0.0049     0.9792     -0.0753      0.6873
          efg efficiency/value 31    -0.2011     0.2779     -0.2208      0.2325
          pps efficiency/value 31    -0.2011     0.2779     -0.2208      0.2325

Mean pre-post correlation by feature group
group
efficiency/value    0.1123
spatial             0.2539


## 13. Repeated-measures models

Event-time panel around migration, a mixed-effects model with a player random
intercept, and an ordinary least squares model with player-clustered standard
errors as the robust fallback.

In [20]:
import statsmodels.formula.api as smf

KEY_FEATURES = ["three_rate", "mean_dist", "efg", "xsv", "make_over_exp"]

ev_rows = []
for r in episodes.itertuples():
    t0 = int(r.destination_season[:4])
    for _, fr in prof[prof.player_id == r.player_id].iterrows():
        et = int(fr.season[:4]) - t0
        if -3 <= et <= 2:
            ev_rows.append({"player_id": r.player_id, "direction": r.direction,
                            "event_time": et,
                            **{f: fr[f] for f in KEY_FEATURES + ["pps"]}})
ev = pd.DataFrame(ev_rows).drop_duplicates(["player_id", "direction", "event_time"])
print(f"Event-time panel: {len(ev)} player-season observations from "
      f"{ev.player_id.nunique()} players\n")
print(ev.groupby("event_time")[["efg", "xsv", "make_over_exp"]].mean().round(4).to_string())

Event-time panel: 300 player-season observations from 90 players

               efg     xsv  make_over_exp
event_time                               
-3          0.5375  1.0347         0.0167
-2          0.5253  1.0357         0.0046
-1          0.5163  1.0315        -0.0006
 0          0.5265  1.0705        -0.0069
 1          0.5238  1.0680        -0.0090
 2          0.5267  1.0590        -0.0014


In [21]:
long_rows = []
for r in episodes.itertuples():
    o_lg, d_lg = o_d_league(r.direction)
    for season, lg, post in [(r.origin_season, o_lg, 0),
                             (r.destination_season, d_lg, 1)]:
        try:
            fr = prof_key.loc[(r.player_id, lg, season)]
        except KeyError:
            continue
        long_rows.append({"player_id": str(r.player_id), "post": post,
                          "direction": r.direction,
                          "age": r.player_age_at_migration,
                          **{f: fr[f] for f in KEY_FEATURES}})
long_df = pd.DataFrame(long_rows).dropna(subset=["age"])

rows = []
for f in KEY_FEATURES:
    d = long_df.dropna(subset=[f])
    rec = {"outcome": f, "n_obs": len(d), "n_players": d.player_id.nunique()}
    try:
        m = smf.mixedlm(f"{f} ~ post * C(direction) + age", d,
                        groups=d["player_id"]).fit()
        rec["mixedlm_post_beta"] = m.params.get("post", np.nan)
        rec["mixedlm_post_p"] = m.pvalues.get("post", np.nan)
        rec["RE_var"] = float(m.cov_re.iloc[0, 0]) if m.cov_re.size else np.nan
    except Exception:
        rec.update(mixedlm_post_beta=np.nan, mixedlm_post_p=np.nan, RE_var=np.nan)
    ols = smf.ols(f"{f} ~ post * C(direction) + age", d).fit(
        cov_type="cluster", cov_kwds={"groups": d["player_id"]})
    rec["ols_post_beta"] = ols.params.get("post", np.nan)
    rec["ols_post_p_clustered"] = ols.pvalues.get("post", np.nan)
    rows.append(rec)
print("Post-migration effect: mixed model vs cluster-robust OLS\n")
print(pd.DataFrame(rows).round(4).to_string(index=False))

Post-migration effect: mixed model vs cluster-robust OLS

      outcome  n_obs  n_players  mixedlm_post_beta  mixedlm_post_p  RE_var  ols_post_beta  ols_post_p_clustered
   three_rate    187         75            -0.0351          0.1427  0.0304        -0.0932                0.1031
    mean_dist    187         75            -0.0910          0.5151  1.3591        -0.5996                0.0943
          efg    187         75            -0.0440          0.0001  0.0017        -0.0472                0.0002
          xsv    187         75             0.0002          0.9903  0.0075         0.0248                0.2400
make_over_exp    187         75            -0.0350          0.0000  0.0013        -0.0465                0.0000


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 14. Exploratory analyses

Two supporting analyses that the manuscript reports as exploratory: shooting
archetypes from within-league standardised clustering, and a grouped
cross-validated random forest predicting post-migration outcomes from the
pre-migration profile.

In [22]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

CLUS = ["rim_rate", "paint_rate", "mid_rate", "corner3_rate", "abovebreak3_rate",
        "three_rate", "mean_dist", "sd_dist", "zone_entropy", "xsv", "xefg"]
prof_c = prof.dropna(subset=CLUS).reset_index(drop=True)
prof_z = prof_c.copy()
for c in CLUS:  # standardise WITHIN league so clusters are archetypes, not leagues
    prof_z[c] = prof_c.groupby("league")[c].transform(
        lambda s: (s - s.mean()) / (s.std(ddof=0) + 1e-9))
Xs = prof_z[CLUS].values

K = 5
km = KMeans(K, n_init=25, random_state=0).fit(Xs)
prof_c["cluster"] = km.labels_
print(f"k = {K}, silhouette = {silhouette_score(Xs, km.labels_):.4f}\n")
print("League composition of each cluster (should be mixed, not league-separated)\n")
print(pd.crosstab(prof_c["cluster"], prof_c["league"]).to_string())
print("\nCluster centroids (original units)\n")
print(prof_c.groupby("cluster")[CLUS].mean().round(3).to_string())

k = 5, silhouette = 0.2330

League composition of each cluster (should be mixed, not league-separated)

league   EuroLeague  NBA
cluster                 
0                83   46
1                45   28
2                59    8
3                26   22
4                95   16

Cluster centroids (original units)

         rim_rate  paint_rate  mid_rate  corner3_rate  abovebreak3_rate  three_rate  mean_dist  sd_dist  zone_entropy    xsv   xefg
cluster                                                                                                                            
0           0.194       0.169     0.111         0.094             0.432       0.526      5.113    2.859         1.341  1.061  0.530
1           0.537       0.276     0.142         0.004             0.041       0.045      2.050    1.922         0.992  1.148  0.574
2           0.022       0.144     0.172         0.117             0.545       0.663      6.217    1.885         1.137  0.957  0.478
3           0.122       

In [23]:
PRE_COLS = [f"{f}__pre" for f in ["three_rate", "rim_rate", "mid_rate", "mean_dist",
                                  "zone_entropy", "efg", "xsv", "make_over_exp"]]
post = prof_pairs.copy()
for f in CORR_FEATURES:
    post[f"{f}__post"] = prof_pairs[f"{f}__post"]

print("Grouped cross-validated random forest (folds grouped by player)\n")
for target in ["xsv__post", "efg__post"]:
    d = post.dropna(subset=PRE_COLS + [target])
    X, y, groups = d[PRE_COLS].values, d[target].values, d["player_id"].values
    gkf = GroupKFold(n_splits=min(5, d["player_id"].nunique()))
    rf = RandomForestRegressor(n_estimators=400, random_state=0)
    pred = cross_val_predict(rf, X, y, groups=groups, cv=gkf)
    print(f"  {target:<12} R2 = {r2_score(y, pred):+.3f}   "
          f"MAE = {mean_absolute_error(y, pred):.4f}   n = {len(d)}")
    rf.fit(X, y)
    imp = pd.Series(rf.feature_importances_, index=PRE_COLS).sort_values(ascending=False)
    print("     top predictors: " + ", ".join(f"{k}({v:.2f})" for k, v in imp.head(4).items()))

Grouped cross-validated random forest (folds grouped by player)



  xsv__post    R2 = +0.392   MAE = 0.0609   n = 31
     top predictors: mid_rate__pre(0.42), xsv__pre(0.28), three_rate__pre(0.10), make_over_exp__pre(0.06)


  efg__post    R2 = +0.016   MAE = 0.0213   n = 31
     top predictors: make_over_exp__pre(0.31), mid_rate__pre(0.22), xsv__pre(0.20), zone_entropy__pre(0.13)


## 15. Paper statistics extraction

Every headline number reported in the paper.

In [24]:
P = pairs[("harmonised", PRIMARY)]
L = pairs[("league_specific", PRIMARY)]


def fmt(df, col):
    d = df[col].dropna()
    lo, hi = boot_ci(d.values)
    return f"{d.mean():+.4f}  [{lo:+.4f}, {hi:+.4f}]  n={len(d)}"


print("=" * 78)
print("SAMPLE")
print("=" * 78)
print(f"  Players in database                     {len(players):>6,}")
print(f"  Cross-league migrants                   {int(players.is_cross_league_migrant.sum()):>6,}")
print(f"  Migration episodes (gap 0 or 1)         {len(episodes):>6,}"
      f"   ({int((episodes.direction=='NBA_to_EL').sum())} NBA-to-EL, "
      f"{int((episodes.direction=='EL_to_NBA').sum())} EL-to-NBA)")
print(f"  Episodes with shot coordinates          {int((episodes.fga_min_side>0).sum()):>6,}")
print(f"  Primary sample (>={PRIMARY} FGA)              {len(P):>6,}"
      f"   ({int((P.direction=='NBA_to_EL').sum())} NBA-to-EL, "
      f"{int((P.direction=='EL_to_NBA').sum())} EL-to-NBA)")
print(f"  ... unique players                      {P.player_id.nunique():>6,}")
print(f"  ... distinct moves                      {P.move_id.nunique():>6,}")

print("\n" + "=" * 78)
print("sxSV MODEL PERFORMANCE (out-of-fold, full population)")
print("=" * 78)
for _, r in model_metrics.iterrows():
    print(f"  {r['spec']:<16} {r['league']:<11} AUC={r['AUC']:.4f}  "
          f"logloss={r['LogLoss']:.4f}  Brier={r['Brier']:.4f}  ECE={r['ECE']:.4f}")

print("\n" + "=" * 78)
print(f"PRE-POST CHANGE, PRIMARY SAMPLE (harmonised sxSV; mean [95% CI])")
print("=" * 78)
for lbl, sub in [("All episodes", P),
                 ("EuroLeague to NBA", P[P.direction == "EL_to_NBA"]),
                 ("NBA to EuroLeague", P[P.direction == "NBA_to_EL"])]:
    print(f"  {lbl}")
    for m, name in [("efg", "eFG "), ("pps", "PPS "), ("sxsv", "sxSV"), ("smoe", "SMOE")]:
        print(f"      {name}  {fmt(sub, f'd_{m}')}")

print("\n" + "=" * 78)
print("sxSV SPECIFICATION COMPARABILITY")
print("=" * 78)
print(f"  Pearson r, change in sxSV               {agree['r_delta_sxSV'].iloc[0]:.4f}")
print(f"  Sign agreement                          {agree['sign_agreement_pct'].iloc[0]:.1f}%")
print(f"  Mean change, harmonised                 {agree['mean_delta_harmonised'].iloc[0]:+.4f}")
print(f"  Mean change, league-specific            {agree['mean_delta_league_specific'].iloc[0]:+.4f}")

print("\n" + "=" * 78)
print("MATCHED-CONTROL COMPARISON OF CHANGES (base matching)")
print("=" * 78)
base = did_tab[did_tab.matching == "base (paper)"]
for _, r in base.iterrows():
    print(f"  {r['direction']:<10} {r['outcome']:<13} {r['estimate']:+8.4f}  "
          f"[{r['ci_lo']:+.4f}, {r['ci_hi']:+.4f}]  n={int(r['n'])}  p={r['wilcoxon_p']:.4g}")

print("\n" + "=" * 78)
print("DEPENDENCE ROBUSTNESS (harmonised, change in eFG)")
print("=" * 78)
for lbl, sub in [("All 31 episodes", P),
                 ("One episode per distinct move",
                  P.sort_values(["move_id", "gap"]).drop_duplicates("move_id")),
                 ("One episode per player",
                  P.sort_values(["player_id", "gap"]).drop_duplicates("player_id"))]:
    s = sub[sub.direction == "EL_to_NBA"]
    print(f"  {lbl:<32} all: {fmt(sub,'d_efg')}")
    print(f"  {'':<32} EL-to-NBA: {fmt(s,'d_efg')}")

print("\n" + "=" * 78)
print("PLAYING TIME AND USAGE (all episodes)")
print("=" * 78)
for v in ["minutes_per_game", "fga_per_game", "fga_per_min", "usage_proxy_per_min"]:
    d = usage[f"d_{v}"].dropna()
    lo, hi = boot_ci(d.values)
    print(f"  {v:<22} {d.mean():+8.4f}  [{lo:+.4f}, {hi:+.4f}]  "
          f"p={wilcoxon(d.values).pvalue:.4g}")
print("=" * 78)

SAMPLE
  Players in database                      3,075
  Cross-league migrants                      402
  Migration episodes (gap 0 or 1)            589   (346 NBA-to-EL, 243 EL-to-NBA)
  Episodes with shot coordinates              83
  Primary sample (>=100 FGA)                  31   (12 NBA-to-EL, 19 EL-to-NBA)
  ... unique players                          11
  ... distinct moves                          19

sxSV MODEL PERFORMANCE (out-of-fold, full population)
  harmonised       NBA         AUC=0.6292  logloss=0.6561  Brier=0.2321  ECE=0.0011
  harmonised       EuroLeague  AUC=0.6506  logloss=0.6361  Brier=0.2238  ECE=0.0010
  league_specific  NBA         AUC=0.6299  logloss=0.6560  Brier=0.2320  ECE=0.0010
  league_specific  EuroLeague  AUC=0.6873  logloss=0.5863  Brier=0.2059  ECE=0.0010

PRE-POST CHANGE, PRIMARY SAMPLE (harmonised sxSV; mean [95% CI])
  All episodes
      eFG   -0.0226  [-0.0489, +0.0038]  n=31
      PPS   -0.0453  [-0.0978, +0.0076]  n=31
      sxSV  -0.0092  [